# Notebook 01: LLM Model Selection and Setup — OFPHIL MSW Interview Assistant

**Project:** `ofphil-classification-v2`  
**Phase:** 05 — Model Selection and Training (`06_llm_interview_assistant`)  
**Runtime Runtime:** Ollama Local Inference Engine (`http://localhost:11434`)  
**Target Hardware Profile:** NVIDIA RTX 3050 (4GB VRAM) / 32GB RAM / Windows 11 / Python 3.12.10  

---

## 1. Research Objectives & Clinical Scope

The **OFPHIL Stage 2 Medical Social Work (MSW) Interview Assistant** is an interactive clinical copilot designed to assist licensed caseworkers during charity subsidy patient intake. It operates in a strictly supportive capacity:

- **Assists with Conversation Flow**: Suggests culturally sensitive, context-aware follow-up questions in English and Filipino/Taglish.
- **Structures Disorganized Narratives**: Organizes free-flowing patient verbal disclosures into a standardized socioeconomic schema.
- **Highlights Information Gaps**: Flags missing financial or documentation items (e.g., unclarified monthly obligations or informal wages) for caseworker review.
- **Strict Governance Boundaries**:
  - The LLM **never** makes eligibility decisions.
  - The LLM **never** assigns charity tiers, discounts, or risk scores.
  - The LLM **never** invents missing information or treats blank answers as negative or zero values.
  - Final decision-making remains exclusively with licensed Medical Social Workers.

This notebook provides the environment validation, runtime connectivity verification, and candidate model registry loading. It does **not** download model weights automatically or initiate inference.

## 2. Pre-trained LLM Candidates & Multi-Model Registry

We evaluate four candidate pre-trained instruct architectures via Ollama:

| Priority | Model Candidate | Ollama Tag | Params | Quantization & Published Size | Native Context Window | License | Target Role & Fit |
|---|---|---|---|---|---|---|---|
| **Initial Testing Candidate** | Qwen3 4B Instruct | `qwen3:4b` | 4.0B | Q4_K_M (~2.5GB) | 32k native / 4k run | Apache 2.0 / Qwen Community | Initial testing candidate; primary choice for local pipeline and prompt validation. |
| **Lightweight Comparison** | Phi-4-mini Instruct | `phi4-mini` | 3.8B | Q4_K_M (~2.5GB) | 128k native / 4k run | MIT | Lightweight comparison; compact reasoning footprint with native function calling. |
| **Higher-Capacity Comparison** | Qwen3 8B Instruct | `qwen3:8b` | 8.2B | Q4_K_M (~5.2GB) | 32k native / 4k run | Apache 2.0 / Qwen Community | Higher-capacity comparison; strong Taglish code-switching; hybrid CPU/GPU offload. |
| **Optional Heavy Comparison** | Gemma 3 12B Instruct | `gemma3:12b` | 12.0B | Q4_K_M (~8.2GB) | 128k native / 4k run | Gemma Terms (NOT Apache 2.0) | Optional heavy comparison; deep reasoning; requires heavy host RAM offloading. |

In [ ]:
# Load Model Registry and Local Runtime Configuration
import sys
import json
from pathlib import Path

# Locate project root dynamically
ROOT_DIR = Path.cwd().resolve()
while ROOT_DIR.name != 'ofphil-classification-v2' and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

CONFIG_DIR = ROOT_DIR / 'experiments' / 'config'
REGISTRY_FILE = CONFIG_DIR / 'llm_model_registry.json'
RUNTIME_FILE = CONFIG_DIR / 'llm_runtime_config.json'

assert REGISTRY_FILE.exists(), f'Registry file missing: {REGISTRY_FILE}'
assert RUNTIME_FILE.exists(), f'Runtime config missing: {RUNTIME_FILE}'

with open(REGISTRY_FILE, 'r', encoding='utf-8') as f:
    model_registry = json.load(f)

with open(RUNTIME_FILE, 'r', encoding='utf-8') as f:
    runtime_config = json.load(f)

print(f"Loaded registry v{model_registry['registry_version']} with {len(model_registry['candidate_models'])} candidate models.")
print(f"Runtime endpoint configured: {runtime_config['local_endpoint']} (Default model: {runtime_config['default_model']})")

## 3. Environment & Python Dependency Check

The cell below inspects the current Python interpreter, operating system, and installed packages (`requests`, `pydantic`, `torch`, `pandas`).

In [ ]:
# Python environment and dependency validation
import platform
import importlib.util

print(f"Python Version: {platform.python_version()} on {platform.system()} {platform.release()}")

required_packages = ['requests', 'pydantic', 'pandas', 'numpy']
optional_packages = ['torch', 'ollama']

print("\nChecking required packages:")
for pkg in required_packages:
    installed = importlib.util.find_spec(pkg) is not None
    status = 'INSTALLED' if installed else 'MISSING'
    print(f"  - {pkg:15s}: [{status}]")

print("\nChecking optional packages:")
for pkg in optional_packages:
    installed = importlib.util.find_spec(pkg) is not None
    status = 'INSTALLED' if installed else 'NOT DETECTED (Optional)'
    print(f"  - {pkg:15s}: [{status}]")

## 4. Ollama Local Service & Model Availability Check

This cell tests connection to the local Ollama daemon at `http://localhost:11434/api/tags` and lists which candidate models are already pulled locally.

In [ ]:
# Ollama local endpoint and installed model inventory check
import requests

endpoint = runtime_config['local_endpoint']
tags_url = f"{endpoint}/api/tags"

print(f"Pinging local Ollama endpoint: {tags_url} ...")
try:
    resp = requests.get(tags_url, timeout=5)
    if resp.status_code == 200:
        data = resp.json()
        installed_models = [m['name'] for m in data.get('models', [])]
        print(f"[ONLINE] Ollama service active. Found {len(installed_models)} local model(s):")
        for m in installed_models:
            print(f"  - {m}")
        
        print("\nCandidate Model Audit against Local Ollama Cache:")
        for cand in model_registry['candidate_models']:
            tag = cand['ollama_model_tag']
            is_present = any(tag in m or m in tag for m in installed_models)
            status_str = "PULLED & AVAILABLE" if is_present else "NOT PULLED (Run: 'ollama pull " + tag + "')"
            print(f"  * {cand['display_name']} ({tag}): [{status_str}]")
    else:
        print(f"[HTTP {resp.status_code}] Ollama returned non-200 status.")
except requests.exceptions.ConnectionError:
    print("[OFFLINE] Ollama service is not running on localhost:11434.")
    print("To start Ollama on Windows, run 'ollama serve' in PowerShell or launch the Ollama tray app.")
except Exception as e:
    print(f"[ERROR] Connectivity check failed: {e}")

## 5. Hardware Compatibility & Memory Footprint Assessment

Our development setup features **4GB VRAM (NVIDIA RTX 3050)** and **32GB System RAM**:

- **`qwen3:4b` & `phi4-mini` (3.8B - 4.0B)**: Full GPU residency (~2.5GB to 3.1GB VRAM). High inference throughput (~35-50 tokens/sec).
- **`qwen3:8b` (8.2B)**: Fits ~24 layers in VRAM (~3.2GB), offloading ~8 layers to System RAM. Smooth interactive generation (~15-25 tokens/sec).
- **`gemma3:12b` (12.0B)**: Fits ~16-18 layers in VRAM, offloading ~6.5GB to System RAM. Feasible but exhibits lower generation speeds (~8-14 tokens/sec).

The cell below prints the detailed memory allocation profiles from the registry.

In [ ]:
# Display hardware compatibility profile for all candidates
import pandas as pd

hw_rows = []
for c in model_registry['candidate_models']:
    fit = c['hardware_fit_assessment']
    hw_rows.append({
        'Model': c['display_name'],
        'Priority': c['priority'],
        'Params': c['parameter_count'],
        'Quant Size': c['quantization'].get('published_model_size_gb') or c['quantization'].get('estimated_size_gb'),
        'VRAM Req (Est)': fit.get('estimated_total_runtime_vram_gb') or fit.get('estimated_vram_usage_gb'),
        'RAM Req (Est)': fit.get('estimated_runtime_overhead_and_kv_gb') or fit.get('estimated_system_ram_usage_gb'),
        'Full GPU Load': 'Probable Partial/Near-Full' if fit.get('full_gpu_fit_4gb_vram') else 'No (Offload)',
        'Feasibility': fit['feasibility']
    })

df_hw = pd.DataFrame(hw_rows)
print(df_hw[['Model', 'Priority', 'Params', 'Quant Size', 'VRAM Req (Est)', 'Full GPU Load']].to_string(index=False))
print("\nDetailed Feasibility Summaries:")
for _, r in df_hw.iterrows():
    print(f"- {r['Model']}: {r['Feasibility']}")

## 6. Manual Setup Checklist for Notebook Execution

Before moving to Notebooks 02 through 05, complete these manual steps in Windows PowerShell:

1. Start Ollama: `ollama serve`
2. Download initial testing candidate: `ollama pull qwen3:4b`
3. Download lightweight comparison: `ollama pull phi4-mini`
4. Download higher-capacity comparison: `ollama pull qwen3:8b`
5. (Optional) Download heavy comparison: `ollama pull gemma3:12b`
6. Re-run Cell 4 above to confirm that candidate tags are marked `[PULLED & AVAILABLE]`.